# Iris: affine score and MLP

This notebook accompanies Exercise 6.1. We begin with the two petal measurements to visualise the decision regions, then use all four measurements.

Values produced by softmax will be called **normalised weights**. They sum to one, but this algebraic property alone does not define an underlying probabilistic model.

## Outline

1. [Data and splitting](#donnees-iris)
2. [Affine score and polyhedral regions](#affine-iris)
3. [Score invariance](#invariance-iris)
4. [MLP and non-affine boundaries](#mlp-iris)
5. [Frequencies and confusion matrices](#confusion-iris)
6. [Returning to all four measurements](#quatre-mesures-iris)

In [ ]:
import jax
import jax.numpy as jnp
import flax
from flax import nnx
import optax
import matplotlib.pyplot as plt
import numpy as np
import sklearn
from sklearn.datasets import load_iris
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

jax.config.update("jax_enable_x64", True)
print(
    f"JAX {jax.__version__}, Flax {flax.__version__}, "
    f"Optax {optax.__version__}, scikit-learn {sklearn.__version__}"
)

<a id="donnees-iris"></a>
## 1. Data and splitting

Load the `Iris` dataset. Initially retain petal length and width. Plot the three species, then construct a deterministic 60 % / 20 % / 20 % split preserving their frequencies.

Centre and scale the features using only the training data. This prevents validation or test data from entering the construction of the machine.

In [ ]:
# Complete this section.

<a id="affine-iris"></a>
## 2. Affine score and polyhedral regions

Train an affine score $x\mapsto Ax+b\in\mathbb R^3$ using cross-entropy. Choose $\lambda\in\{0,10^{-4},10^{-3},10^{-2}\}$ from the validation error.

Plot the decision regions. For a class $a$, verify on the grid the inequalities
$$(A_a-A_c)x\geq b_c-b_a,\qquad c\ne a.$$

In [ ]:
class ScoreAffine(nnx.Module):
    def __init__(self, n_entrees, n_classes, *, rngs):
        self.sortie = nnx.Linear(n_entrees, n_classes, rngs=rngs)

    def __call__(self, x):
        return self.sortie(x)


class MLP(nnx.Module):
    def __init__(self, n_entrees, largeur, n_classes, *, rngs):
        self.W1 = nnx.Linear(n_entrees, largeur, rngs=rngs)
        self.W2 = nnx.Linear(largeur, n_classes, rngs=rngs)

    def __call__(self, x):
        return self.W2(nnx.relu(self.W1(x)))


def norme_parametres(machine):
    etat = nnx.state(machine, nnx.Param)
    return sum(jnp.sum(feuille**2) for feuille in jax.tree.leaves(etat))


def perte(machine, x, z, lamb=0.0):
    scores = machine(x)
    ce = optax.softmax_cross_entropy_with_integer_labels(scores, z).mean()
    return ce + 0.5 * lamb * norme_parametres(machine)


@nnx.jit
def pas(machine, optimizer, x, z, lamb):
    valeur, gradient = nnx.value_and_grad(perte)(machine, x, z, lamb)
    optimizer.update(machine, gradient)
    return valeur


def entrainer(machine, x, z, lamb=0.0, alpha=0.03, iterations=700):
    optimizer = nnx.Optimizer(machine, optax.adam(alpha), wrt=nnx.Param)
    for _ in range(iterations):
        valeur = pas(machine, optimizer, x, z, lamb)
    return float(valeur)


def frequence_erreur(machine, x, z):
    predictions = np.asarray(jnp.argmax(machine(jnp.asarray(x)), axis=1))
    return float(np.mean(predictions != np.asarray(z)))


def nombre_parametres(machine):
    return sum(feuille.size for feuille in jax.tree.leaves(nnx.state(machine, nnx.Param)))

In [ ]:
# Complete this section.

<a id="invariance-iris"></a>
## 3. Score invariance

For three flowers, display the scores, normalised weights and predicted class. Then add the same constant to all three scores and verify the invariance of softmax and the decision.

In [ ]:
# Complete this section.

<a id="mlp-iris"></a>
## 4. MLP and non-affine boundaries

Train MLPs with one hidden layer of width 8 or 16. Choose the width and initialisation using validation data, then compare the selected boundary with that of the affine score.

In [ ]:
# Complete this section.

<a id="confusion-iris"></a>
## 5. Frequencies and confusion matrices

Compare error frequencies on all three sets. Construct raw and normalised confusion matrices on the test data.

In [ ]:
# Complete this section.

<a id="quatre-mesures-iris"></a>
## 6. Returning to all four measurements

Repeat the experiment with all four measurements. Planar region visualisation is no longer available, but frequencies and confusion matrices remain defined.

In [ ]:
# Complete this section.

## Summary

The affine score produces convex polyhedral regions. The MLP can curve or split these regions, but this additional freedom is useful only if the data require it. The confusion matrix shows that errors here concentrate between *versicolor* and *virginica*.